# Adquisición de datos

En este notebook se configura la frecuencia de muestreo del sensor, se compara la frecuencia configurada con la que realmente llega a la computadora y se graban señales en archivos CSV, cada una con la etiqueta de la actividad que se realizó.

## Parámetros

`CARPETA` es donde se guardan las grabaciones. Para las grabaciones de cada equipo se cambia a `datos/equipo_A` o `datos/equipo_B`.

In [ ]:
DIRECCION = "FA:81:12:D8:44:D0"   # dirección del sensor del equipo
CARPETA = "datos/demo"

## El módulo wt9011

Las funciones de `01_conexion` están reunidas en el archivo `wt9011.py`, junto con la clase `SensorWT`, que mantiene la conexión abierta y ofrece estas operaciones:

| Método | Qué hace |
|---|---|
| `conectar()`, `desconectar()` | abre y cierra la conexión |
| `bateria()`, `temperatura()` | voltaje de la batería y temperatura interna |
| `frecuencia()` | frecuencia de muestreo configurada, en Hz |
| `configurar_frecuencia(hz)` | cambia la frecuencia de muestreo |
| `grabar(segundos)` | graba y regresa una tabla con una fila por muestra |

In [ ]:
import asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wt9011 import SensorWT, porcentaje_bateria, agregar_tiempo, guardar_csv

sensor = SensorWT(DIRECCION)
await sensor.conectar()

voltaje = await sensor.bateria()
hz = await sensor.frecuencia()
print(f"Batería: {voltaje:.2f} V, aproximadamente {porcentaje_bateria(voltaje)} %")
print(f"Frecuencia configurada: {f'{hz} Hz' if hz else 'distinta de 10, 20, 50 y 100 Hz'}")

## Elección de la frecuencia de muestreo

Según el teorema de muestreo de Nyquist, para representar una señal sin perder información la frecuencia de muestreo debe ser mayor que el doble de la frecuencia más alta contenida en la señal. Si no se cumple aparece aliasing: las componentes rápidas aparecen en las muestras como si fueran lentas, y después ya no se pueden distinguir de las componentes lentas verdaderas.

En el ejemplo, una senoidal de 9 Hz se muestrea a 10 Hz, muy por debajo de los 18 Hz que pide Nyquist. Las muestras quedan exactamente sobre una senoidal de 1 Hz, que es la diferencia entre las dos frecuencias.

In [ ]:
f_senal = 9     # Hz
fs_baja = 10    # Hz

t_continuo = np.linspace(0, 1, 2000)
t_muestras = np.arange(0, 1, 1 / fs_baja)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_continuo, np.sin(2 * np.pi * f_senal * t_continuo), lw=0.8, alpha=0.5, label="señal de 9 Hz")
ax.plot(t_continuo, -np.sin(2 * np.pi * (fs_baja - f_senal) * t_continuo), "--", lw=1.2, label="alias de 1 Hz")
ax.plot(t_muestras, np.sin(2 * np.pi * f_senal * t_muestras), "o", label="muestras a 10 Hz")
ax.set(xlabel="Tiempo [s]", ylabel="Amplitud")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.3)
plt.show()

En los movimientos humanos, los pasos al caminar tienen una frecuencia de 1 a 3 Hz y el temblor fisiológico de la mano está entre 8 y 12 Hz. Casi toda la información de las actividades cotidianas queda por debajo de 20 Hz, así que 50 Hz es suficiente. Muestrear más rápido no agrega información útil: produce más datos, gasta más batería y satura más el enlace Bluetooth.

El sensor ofrece 10, 20, 50 y 100 Hz.

In [ ]:
FRECUENCIA = 50   # Hz

await sensor.configurar_frecuencia(FRECUENCIA)
print(f"Frecuencia configurada: {await sensor.frecuencia()} Hz")

## Frecuencia configurada y frecuencia real

Se graban 5 segundos. La columna `t_llegada` indica en qué momento llegó cada paquete a la computadora, en segundos desde el primero.

In [ ]:
DURACION = 5

df = await sensor.grabar(DURACION)
print(f"Muestras: {len(df)}, esperadas: {FRECUENCIA * DURACION}")
df.round(3).head()

Si cada muestra llegara en cuanto se mide, el tiempo entre paquetes consecutivos sería siempre 1/fs, es decir 20 ms a 50 Hz. El histograma muestra otra cosa.

In [ ]:
dt = np.diff(df["t_llegada"]) * 1000   # ms

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(dt, bins=60)
ax.axvline(1000 / FRECUENCIA, color="k", ls="--", lw=1, label=f"1/fs = {1000 / FRECUENCIA:.0f} ms")
ax.set(xlabel="Tiempo entre paquetes consecutivos [ms]", ylabel="Número de paquetes")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f"Paquetes que llegaron casi al mismo tiempo que el anterior (< 1 ms): {np.mean(dt < 1):.0%}")

Buena parte de los paquetes llega prácticamente al mismo tiempo que el anterior, y el resto después de una espera mayor que 20 ms. El Bluetooth no envía cada muestra en cuanto se mide: las acumula y las manda en grupos, varias por notificación. Por eso el tiempo de llegada no es el tiempo en que se tomó la muestra.

El sensor sí mide a intervalos regulares, así que el tiempo correcto de la muestra n es t = n / fs. Para estimar fs se grafica el tiempo de llegada contra el número de muestra y se ajusta una recta: la pendiente es el periodo de muestreo. A la derecha se ve el detalle de las primeras muestras, donde se notan los escalones de cada grupo.

In [ ]:
n = np.arange(len(df))
periodo, ordenada = np.polyfit(n, df["t_llegada"], 1)
fs_real = 1 / periodo

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, hasta in [(ax1, len(df)), (ax2, 30)]:
    ax.plot(n[:hasta], df["t_llegada"][:hasta], ".", ms=4, label="llegada")
    ax.plot(n[:hasta], ordenada + periodo * n[:hasta], lw=1, label="recta ajustada")
    ax.set(xlabel="Número de muestra", ylabel="Tiempo de llegada [s]")
    ax.grid(alpha=0.3)
ax1.legend()
ax2.set_title("Primeras 30 muestras")
plt.show()

print(f"Periodo de muestreo: {periodo * 1000:.2f} ms")
print(f"Frecuencia real:     {fs_real:.2f} Hz (configurada: {FRECUENCIA} Hz)")

Si se pierden paquetes durante la transmisión, la gráfica muestra saltos hacia arriba y el número de muestras queda por debajo del esperado.

La función `agregar_tiempo` hace este mismo ajuste y agrega a la tabla la columna `t`, el eje de tiempo uniforme que se usará en adelante.

In [ ]:
fs = agregar_tiempo(df)
print(f"fs = {fs:.2f} Hz")
df.round(3).head()

El mismo experimento para las cuatro frecuencias disponibles, 3 segundos cada una. Al final el sensor regresa a la frecuencia elegida arriba.

In [ ]:
filas = []
for hz in [10, 20, 50, 100]:
    await sensor.configurar_frecuencia(hz)
    await asyncio.sleep(0.5)
    prueba = await sensor.grabar(3)
    filas.append({"configurada [Hz]": hz,
                  "muestras en 3 s": len(prueba),
                  "real [Hz]": round(agregar_tiempo(prueba), 2)})

await sensor.configurar_frecuencia(FRECUENCIA)
pd.DataFrame(filas)

La frecuencia real no coincide exactamente con la configurada porque el reloj interno del sensor tiene una pequeña desviación. En el sensor usado para preparar la clase quedó alrededor de 0.6 % por debajo en todas las frecuencias. La diferencia es pequeña, pero en una grabación larga se acumula: en 10 minutos a 50 Hz son unas 180 muestras. Por eso en adelante se usa la fs estimada de cada grabación y no el valor configurado.

## Grabación de una actividad

Cada grabación corresponde a una sola actividad y se guarda como `<etiqueta>_<número>.csv`; el número se asigna solo. El archivo contiene la columna `t`, la columna `t_llegada` y los nueve canales del sensor.

El sensor debe ir siempre en el mismo lugar del cuerpo y con la misma orientación, por ejemplo en la muñeca con el lado del LED hacia afuera. Si cambia de posición entre grabaciones, las señales de una misma actividad dejan de parecerse, y eso después afecta a cualquier modelo que se entrene con ellas.

In [ ]:
ETIQUETA = "caminar"
DURACION = 10   # segundos

print("Grabando...")
df = await sensor.grabar(DURACION)
fs = agregar_tiempo(df)
ruta = guardar_csv(df, CARPETA, ETIQUETA)
print(f"{len(df)} muestras a {fs:.1f} Hz, guardadas en {ruta}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for eje in ["ax", "ay", "az"]:
    ax1.plot(df["t"], df[eje], lw=0.8, label=eje)
for eje in ["gx", "gy", "gz"]:
    ax2.plot(df["t"], df[eje], lw=0.8, label=eje)
ax1.set(ylabel="Aceleración [g]", title=ETIQUETA)
ax2.set(ylabel="Velocidad angular [°/s]", xlabel="Tiempo [s]")
for ax in (ax1, ax2):
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
plt.show()

## Varias repeticiones

Para el conjunto de datos del final de la clase cada equipo graba tres actividades, con cinco repeticiones de cada una. Entre repeticiones hay una pausa para que la persona se acomode y empiece la actividad antes de que arranque la grabación.

In [ ]:
ETIQUETA = "quieto"
REPETICIONES = 5
DURACION = 10   # segundos
PAUSA = 5       # segundos antes de cada grabación

for i in range(1, REPETICIONES + 1):
    print(f"Repetición {i} de {REPETICIONES}, empieza en {PAUSA} s")
    await asyncio.sleep(PAUSA)
    print("  grabando...")
    df = await sensor.grabar(DURACION)
    fs = agregar_tiempo(df)
    ruta = guardar_csv(df, CARPETA, ETIQUETA)
    print(f"  {len(df)} muestras a {fs:.1f} Hz, {ruta}")

Grabaciones que hay en la carpeta, por etiqueta:

In [ ]:
from pathlib import Path

archivos = sorted(Path(CARPETA).glob("*.csv"))
etiquetas = [a.stem.rsplit("_", 1)[0] for a in archivos]   # "caminar_03" -> "caminar"
pd.Series(etiquetas).value_counts()

## Desconexión

In [ ]:
await sensor.desconectar()

En `03_visualizacion` se cargan estos archivos y se interpretan las señales de cada actividad.